In [ ]:
## 학습한 모델을 불러와서, 
## 1) classification accuracy를 본다. (binary classification으로 정의해서, recall, precision, f1 보기)

## 2) locating accuracy를 본다. 
## EPR snli(또는 mnli) 데이터셋에 대해서 locate 코드를 실행하고, 실행한 결과로 token 단위에서 locate을 했는지, 안했는지 결과 값을 반환 받는다.
## 이 값을 tok2word 라는 dictionary에 통과시키면, word 단위에서 locate을 했는지, 안했는지 결과 값도 반환 받을 수 있다.

## 실제 정답은 token 단위로도 있고, word 단위로도 있다. (character 단위는 생략)

## token 단위 정답값이랑 비교하면 -> token 단위에서 mean recall, mean precision, mean average precision, mean reciprocal rank를 구할 수 있다.
## word 단위 정답값이랑 비교하면 -> word 단위에서 .. 를 구할 수 있다.

In [1]:
import os
import re
import random
import json
from typing import List

import wandb
import tqdm
import pandas as pd
from sklearn.metrics import confusion_matrix as cm
import numpy as np
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
)
from scipy import stats

from new_module.locate.new_locate_utils import LocateMachine
from new_module.em_training.nli.models import EncoderModel
from new_module.em_training.nli.data_handling import load_nli_data, load_nli_test_data, NLI_Dataset, NLI_DataLoader

# Define dataset and dataloader
class NLIDataset(Dataset):
    def __init__(self, dataset, indexes):
        self.dataset = dataset
        self.indexes = indexes
        
    def __getitem__(self, idx):
        return self.dataset[self.indexes[idx]]
    
    def __getitems__(self, idxes:List[int]):
        return [self.dataset[self.indexes[j]] for j in idxes]
    
    def __len__(self):
        return len(self.indexes)
    
def collate_fn_with_labels(examples):
    return [(x['premise'], x['hypothesis'], x['gold_label']) for x in examples]

def collate_fn(examples):
    return [(x['premise'], x['hypothesis']) for x in examples]

def get_word_pred_indexes_from_token_pred_indexes(row, tok2word_col: str, token_pred_indexes_col: str):
    words_indexes = []
    for id in row[token_pred_indexes_col]:
        words_indexes.append(row[tok2word_col][str(id)])
    return sorted(list(set(words_indexes)))

def get_word_pred_from_word_pred_indexes(row, words_col: str, word_pred_indexes_col: str):
    return [1 if i in row[word_pred_indexes_col] else 0 for i in range(len(row[words_col]))]

def get_pred_scores_word(row,words_col:str, token_pred_scores_col: str, word2tok_col:str, method='sum'):
    return_list=[]
    if method=='sum':
        func=np.sum
    elif method=='max':
        func=np.max
    elif method=='mean':
        func=np.mean
    for word_id in range(len(row[words_col])):
        return_list.append(func(np.array(row[token_pred_scores_col])[row[word2tok_col][str(word_id)]]))
    return return_list

def apply_ap(row, binary_labels_col:str, pred_scores_col:str):

    if sum(row[binary_labels_col])==0:
        return np.nan
    else:
        return average_precision_score(row[binary_labels_col],row[pred_scores_col])
    
def apply_precision(row, binary_labels_col:str, binary_preds_col:str):

    return precision_score(row[binary_labels_col],row[binary_preds_col], zero_division=np.nan)

def apply_recall(row, binary_labels_col:str, binary_preds_col:str):

    return recall_score(row[binary_labels_col],row[binary_preds_col], zero_division=np.nan)

def rr(out, labels, k = 6): #implement mean reciprocal rank
    idx_array = stats.rankdata(-out, axis=-1, method='min')
    # print(idx_array)
    labels = np.where(labels==1)[0].astype(int)
    # print(labels)
    rank = np.take_along_axis(idx_array, labels, axis=-1)
    # print(rank)
    rr=1/rank.min() if rank.min() <= k else 0.
    return rr

def get_rr(row, binary_labels_col:str, pred_scores_col:str):
    """suffix should start with _"""
    if sum(row[binary_labels_col])==0:
        return np.nan
    else:
        return rr(np.array(row[pred_scores_col]),np.array(row[binary_labels_col]))
    

device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
batch_size = 64
dev_data = pd.read_json('/data/hyeryung/mucoco/new_module/data/EPR/text_file/snli_annotation/snli_locate_labels.jsonl', lines=True)
nli_dataset = dev_data.to_dict(orient="records")

all_indexes = list(range(len(nli_dataset)))
all_data = NLIDataset(nli_dataset, all_indexes)
all_dataloader = DataLoader(all_data, batch_size=batch_size, collate_fn = collate_fn_with_labels, shuffle=False)

contradiction_indexes = [i for i, x in enumerate(nli_dataset) if x['gold_label'] == 'contradiction']
contradiction_data = NLIDataset(nli_dataset, contradiction_indexes)
contra_dataloader = DataLoader(contradiction_data, batch_size=batch_size, collate_fn = collate_fn, shuffle=False)
print(f"# of all data: {len(all_data)}")
print(f"# of contradiction data: {len(contradiction_data)}")

# of all data: 100
# of contradiction data: 32


# Binary classifier model

In [198]:
run_id = 'hayleyson/nli_energynet/ovp9pnpb'
criterion = 'loss'

In [199]:
## load config
api = wandb.Api()
run = api.run(run_id)
config = run.config
config['device'] = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## load model
model = EncoderModel(config)
model = model.to(config['device'])

if run_id.split('/')[-1] == 'ovp9pnpb': ## exception case handling
    model_path = '/data/hyeryung/loc_edit/' + config['model_path'].split('.pth')[0] + f'_{criterion}.pth'
elif run_id.split('/')[-1] == '8a9jdnfg': ## exception case handling
    model_path = '/data/hyeryung/loc_edit/' + '/'.join(config['model_path'].split('/')[1:]).split('.pth')[0] + f'_{criterion}.pth'
else:
    model_path = config['model_path'].split('.pth')[0] + f'_{criterion}.pth'
print(f"model_path: {model_path}")

try:
    model.load_state_dict(torch.load(model_path,weights_only=True))
    config['model_path'] = model_path
except Exception as e:
    model.load_state_dict(torch.load(config['model_path'],weights_only=True))
    
## define tokenizer
tokenizer = model.tokenizer

/data/hyeryung/.conda/envs/loc-edit/lib/python3.8/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at roberta-large were not used when initializing RobertaModel: ['lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.weight', 'lm_head.dense.bias']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of

model_path: /data/hyeryung/loc_edit/models/nli/roberta_large_snli_mnli_anli_train_dev_with_finegrained_binary_labels_binary_cross_entropy/1730009174/best_model_loss.pth


## Calculate classification accuracy 

In [200]:
## calculate accuracy 
num_correct = 0
labels_binary_all = []
pred_c_binary_all = []
for batch in tqdm.tqdm(all_dataloader):
    premises = [x[0] for x in batch]
    hypotheses = [x[1] for x in batch]
    labels = [x[2] for x in batch]
    labels_binary = torch.IntTensor([1 if lab != 'contradiction' else 0 for lab in labels])
    
    if (config['energynet'].get('input_form', 'x_only') == 'x_only'):  
        sequences = [tokenizer.bos_token + p + tokenizer.sep_token + h + tokenizer.eos_token for p, h in zip(premises, hypotheses)]
    tokenized_sequences = tokenizer(sequences, add_special_tokens=False,padding=True, truncation=True, return_tensors='pt')
    tokenized_sequences = tokenized_sequences.to(config['device'])
    logits, hidden_states = model(**tokenized_sequences)
    
    pred_c_binary = logits.argmax(axis=-1).cpu()
    
    num_correct += (labels_binary == pred_c_binary).sum().item()
    pred_c_binary_all.extend(pred_c_binary.tolist())
    labels_binary_all.extend(labels_binary.tolist())
acc = num_correct / len(all_data)
print(acc)


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  8.88it/s]

0.96


In [201]:
dev_data['gold_label_binary'] = labels_binary_all
dev_data['pred_binary'] = pred_c_binary_all
if criterion is not None:
    dev_data[['pairID', 'gold_label_binary', 'pred_binary']].to_csv(os.path.join(os.path.dirname(model_path), f'EPR_classification_result_{criterion}.csv'),index=False)
else:
    dev_data[['pairID', 'gold_label_binary', 'pred_binary']].to_csv(os.path.join(os.path.dirname(model_path), 'EPR_classification_result.csv'),index=False)

## Calculate locate performance

In [214]:
# ARGS
use_energy_for_gradient = True
max_num_tokens = 100

In [215]:
locator = LocateMachine(model, model.tokenizer, 'nli')

pred_all = []
pred_indexes_all = []
pred_scores_all = []
masked_texts_all = []
for batch in tqdm.tqdm(contra_dataloader):        
    
    premises = [x[0] for x in batch]
    hypotheses = [x[1] for x in batch]
    if (config['energynet'].get('input_form', 'x_only') == 'x_only'):  
        sequences = [tokenizer.bos_token + p + tokenizer.sep_token + h + tokenizer.eos_token for p, h in zip(premises, hypotheses)]
    tokenized_sequences = tokenizer(sequences, add_special_tokens=False,padding=True, truncation=True, return_tensors='pt')
    tokenized_sequences = tokenized_sequences.to(config['device'])
    
    masked_texts, scores = locator.locate_main(tokenized_sequences, 
                                 'grad_norm', 
                                 max_num_tokens = max_num_tokens, 
                                 unit="word", 
                                 label_id=config['energynet']['energy_col'], 
                                 tokenized_input=True,
                                 return_scores=True,
                                 use_energy=use_energy_for_gradient)
    
    tokenized_result = tokenizer(masked_texts, add_special_tokens=False)
    hypotheses_input_ids = [x[x.index(2)+1:-1] for x in tokenized_result['input_ids']]
    hypotheses_scores = [x.tolist()[y.index(2)+1:len(y)-1] for x, y in zip(scores, tokenized_result['input_ids'])]
    
    pred = [[1 if y == tokenizer.mask_token_id else 0 for y in x] for x in hypotheses_input_ids]
    pred_indexes = [np.where(np.array(x) == tokenizer.mask_token_id)[0].tolist() for x in hypotheses_input_ids]
    
    pred_all.extend(pred)
    pred_indexes_all.extend(pred_indexes)
    pred_scores_all.extend(hypotheses_scores)
    masked_texts_all.extend(masked_texts)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.09it/s]


In [216]:
contra_data = dev_data.loc[dev_data['gold_label'] == 'contradiction',].copy()
contra_data['hypothesis_token_pred_binary'] = pred_all
contra_data['hypothesis_token_pred_indexes'] = pred_indexes_all
contra_data['hypothesis_token_pred_scores'] = pred_scores_all

In [217]:
contra_data['hypothesis_word_pred_indexes'] = contra_data.apply(lambda x: get_word_pred_indexes_from_token_pred_indexes(x, 'hypothesis_tok2word', 'hypothesis_token_pred_indexes'), axis=1)
contra_data['hypothesis_word_pred_binary'] = contra_data.apply(lambda x: get_word_pred_from_word_pred_indexes(x, 'hypothesis_words', 'hypothesis_word_pred_indexes'), axis=1)
contra_data['hypothesis_word_pred_scores'] = contra_data.apply(lambda x: get_pred_scores_word(x, 'hypothesis_words', 'hypothesis_token_pred_scores', 'hypothesis_word2tok', method='max'),axis=1)

In [218]:
# ## token에서 word로 mapping이 맞게 되었는지 확인할 필요가 있다. --> sample 10개 가량 확인했을 때 문제 없음

# sample_id = random.choice(contra_data.index.tolist())
# sample_tokens = contra_data.loc[sample_id, 'hypothesis_tokens'] 
# sample_tokens = [tokenizer.decode(x) for x in sample_tokens]
# sample_words = contra_data.loc[sample_id, 'hypothesis_words'] 
# sample_tok2word = contra_data.loc[sample_id, 'hypothesis_tok2word'] 

# print(sample_tokens)
# print(sample_words)
# print(sample_tok2word)

In [219]:
# ## 혹시 tokens 컬럼과 token_pred 컬럼의 길이가 다른 경우가 있는지 확인 --> 없음
# (contra_data['hypothesis_tokens'].apply(len) != contra_data['hypothesis_token_pred_binary'].apply(len) ).sum()

In [220]:
# ## words_scores 컬럼의 값의 길이가 words_list와 같은지 체크 --> 없음
# (contra_data['hypothesis_word_scores'].apply(len) != contra_data['hypothesis_words'].apply(len) ).sum()

In [221]:
# ### Calculate Word-level Metrics
contra_data['ap_word']=contra_data.apply(lambda x: apply_ap(x,"hypothesis_word_labels_binary", "hypothesis_word_pred_scores"),axis=1)
contra_data['rr_word']=contra_data.apply(lambda x: get_rr(x,"hypothesis_word_labels_binary", "hypothesis_word_pred_scores"),axis=1)
contra_data['precision_word']=contra_data.apply(lambda x: apply_precision(x,"hypothesis_word_labels_binary", "hypothesis_word_pred_binary"),axis=1)
contra_data['recall_word']=contra_data.apply(lambda x: apply_recall(x,"hypothesis_word_labels_binary", "hypothesis_word_pred_binary"),axis=1)

## Summary metric
mrr = contra_data['rr_word'].mean()
map_score =  contra_data['ap_word'].mean()
precision = contra_data['precision_word'].mean()
recall = contra_data['recall_word'].mean()

In [222]:
# ### Calculate Token-level Metrics
contra_data['ap_tokens']=contra_data.apply(lambda x: apply_ap(x,"hypothesis_token_labels_binary", "hypothesis_token_pred_scores"),axis=1)
contra_data['rr_tokens']=contra_data.apply(lambda x: get_rr(x,"hypothesis_token_labels_binary", "hypothesis_token_pred_scores"),axis=1)
contra_data['precision_tokens']=contra_data.apply(lambda x: apply_precision(x,"hypothesis_token_labels_binary", "hypothesis_token_pred_binary"),axis=1)
contra_data['recall_tokens']=contra_data.apply(lambda x: apply_recall(x,"hypothesis_token_labels_binary", "hypothesis_token_pred_binary"),axis=1)

## Summary metric
mrr_tokens = contra_data['rr_tokens'].mean()
map_score_tokens =  contra_data['ap_tokens'].mean()
precision_tokens = contra_data['precision_tokens'].mean()
recall_tokens = contra_data['recall_tokens'].mean()

In [223]:
if criterion is not None:
    contra_data[['pairID', 'hypothesis_word_pred_binary', 'hypothesis_word_pred_scores', 'hypothesis_token_pred_binary', 'hypothesis_token_pred_scores']].to_json(os.path.join(os.path.dirname(model_path), f'EPR_locate_result_{criterion}_{"energy" if use_energy_for_gradient else "proba"}.jsonl'), lines=True, orient='records')
else:
    contra_data[['pairID', 'hypothesis_word_pred_binary', 'hypothesis_word_pred_scores', 'hypothesis_token_pred_binary', 'hypothesis_token_pred_scores']].to_json(os.path.join(os.path.dirname(model_path), f'EPR_locate_result_{"energy" if use_energy_for_gradient else "proba"}.jsonl'), lines=True, orient='records')

In [224]:
metrics_path = os.path.join(os.path.dirname(model_path), 'EPR_metrics.csv')

if not os.path.exists(metrics_path):
    pd.DataFrame({'run_id': [run_id],
                  'criterion': [criterion],
                  'use_energy_for_gradient':[use_energy_for_gradient],
                'classification_accuracy': [acc],
                'mrr_words': [mrr],
                'map_words': [map_score],
                'mean precision_words': [precision],
                'mean recall_words': [recall],
                'mrr_tokens': [mrr_tokens], 
                'map_tokens': [map_score_tokens],
                'mean precision_tokens': [precision_tokens],
                'mean recall_tokens': [recall_tokens]}).to_csv(metrics_path)
else:
    with open(metrics_path, 'a') as f:
        f.write(f"{run_id},{criterion},{use_energy_for_gradient},{acc},{mrr},{map_score},{precision},{recall},{mrr_tokens},{map_score_tokens},{precision_tokens},{recall_tokens}\n")
   

In [225]:
print("Metrics evaluated at words level")
print(f"mrr: {mrr:.4f}")
print(f"map: {map_score:.4f}")
print(f"mean precision: {precision:.4f}")
print(f"mean recall: {recall:.4f}")

print("Metrics evaluated at tokens level")
print(f"mrr: {mrr_tokens:.4f}")
print(f"map: {map_score_tokens:.4f}")
print(f"mean precision: {precision_tokens:.4f}")
print(f"mean recall: {recall_tokens:.4f}")

Metrics evaluated at words level
mrr: 0.7011
map: 0.6288
mean precision: 0.4375
mean recall: 0.4807
Metrics evaluated at tokens level
mrr: 0.8360
map: 0.7241
mean precision: 0.6432
mean recall: 0.5538


# Multiclass classifier model

In [98]:
run_id = 'hayleyson/nli_energynet/a9v4hdlt'
criterion = 'pearsonr'

In [99]:
## load config
api = wandb.Api()
run = api.run(run_id)
config = run.config
config['device'] = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## load model
model = EncoderModel(config)
model = model.to(config['device'])

if run_id.split('/')[-1] == 'ovp9pnpb': ## exception case handling
    model_path = '/data/hyeryung/loc_edit/' + config['model_path'].split('.pth')[0] + f'_{criterion}.pth'
elif run_id.split('/')[-1] == '8a9jdnfg': ## exception case handling
    model_path = '/data/hyeryung/loc_edit/' + '/'.join(config['model_path'].split('/')[1:]).split('.pth')[0] + f'_{criterion}.pth'
else:
    model_path = config['model_path'].split('.pth')[0] + f'_{criterion}.pth'
print(f"model_path: {model_path}")

try:
    model.load_state_dict(torch.load(model_path,weights_only=True))
    config['model_path'] = model_path
except Exception as e:
    model.load_state_dict(torch.load(config['model_path'],weights_only=True))
    
## define tokenizer
tokenizer = model.tokenizer

/data/hyeryung/.conda/envs/loc-edit/lib/python3.8/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at roberta-large were not used when initializing RobertaModel: ['lm_head.layer_norm.weight', 'lm_head.dense.bias', 'lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.dense.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of

model_path: /data/hyeryung/loc_edit/models/nli/roberta_large_snli_mnli_anli_train_dev_with_finegrained_original_labels_cross_entropy/1730162466/best_model_pearsonr.pth


## Calculate classification accuracy 

In [101]:
## calculate accuracy 
num_correct = 0
labels_binary_all = []
pred_c_binary_all = []
for batch in tqdm.tqdm(all_dataloader):
    premises = [x[0] for x in batch]
    hypotheses = [x[1] for x in batch]
    labels = [x[2] for x in batch]
    labels_binary = torch.IntTensor([1 if lab != 'contradiction' else 0 for lab in labels])
    
    if (config['energynet'].get('input_form', 'x_only') == 'x_only'):  
        sequences = [tokenizer.bos_token + p + tokenizer.sep_token + h + tokenizer.eos_token for p, h in zip(premises, hypotheses)]
    tokenized_sequences = tokenizer(sequences, add_special_tokens=False,padding=True, truncation=True, return_tensors='pt')
    tokenized_sequences = tokenized_sequences.to(config['device'])
    logits, hidden_states = model(**tokenized_sequences)
    
    pred_c = logits.argmax(axis=-1).cpu()
    pred_c_binary = torch.IntTensor([1 if pred != 2 else 0 for pred in pred_c])
    
    num_correct += (labels_binary == pred_c_binary).sum().item()
    pred_c_binary_all.extend(pred_c_binary.tolist())
    labels_binary_all.extend(labels_binary.tolist())
acc = num_correct / len(all_data)
print(acc)


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 17.02it/s]

0.95


In [102]:
dev_data['gold_label_binary'] = labels_binary_all
dev_data['pred_binary'] = pred_c_binary_all
if criterion is not None:
    dev_data[['pairID', 'gold_label_binary', 'pred_binary']].to_csv(os.path.join(os.path.dirname(model_path), f'EPR_classification_result_{criterion}.csv'),index=False)
else:
    dev_data[['pairID', 'gold_label_binary', 'pred_binary']].to_csv(os.path.join(os.path.dirname(model_path), 'EPR_classification_result.csv'),index=False)

## Calculate locate performance

In [112]:
# ARGS
use_energy_for_gradient = False
max_num_tokens = 100
multiclass_ce = True

In [113]:
locator = LocateMachine(model, model.tokenizer, 'nli')

pred_all = []
pred_indexes_all = []
pred_scores_all = []
masked_texts_all = []
for batch in tqdm.tqdm(contra_dataloader):        
    
    premises = [x[0] for x in batch]
    hypotheses = [x[1] for x in batch]
    if (config['energynet'].get('input_form', 'x_only') == 'x_only'):  
        sequences = [tokenizer.bos_token + p + tokenizer.sep_token + h + tokenizer.eos_token for p, h in zip(premises, hypotheses)]
    tokenized_sequences = tokenizer(sequences, add_special_tokens=False,padding=True, truncation=True, return_tensors='pt')
    tokenized_sequences = tokenized_sequences.to(config['device'])
    
    masked_texts, scores = locator.locate_main(tokenized_sequences, 
                                 'grad_norm', 
                                 max_num_tokens = max_num_tokens, 
                                 unit="word", 
                                 label_id=config['energynet']['energy_col'], 
                                 tokenized_input=True,
                                 return_scores=True,
                                 use_energy=use_energy_for_gradient,
                                 multiclass_ce=multiclass_ce)
    
    tokenized_result = tokenizer(masked_texts, add_special_tokens=False)
    hypotheses_input_ids = [x[x.index(2)+1:-1] for x in tokenized_result['input_ids']]
    hypotheses_scores = [x.tolist()[y.index(2)+1:len(y)-1] for x, y in zip(scores, tokenized_result['input_ids'])]
    
    pred = [[1 if y == tokenizer.mask_token_id else 0 for y in x] for x in hypotheses_input_ids]
    pred_indexes = [np.where(np.array(x) == tokenizer.mask_token_id)[0].tolist() for x in hypotheses_input_ids]
    
    pred_all.extend(pred)
    pred_indexes_all.extend(pred_indexes)
    pred_scores_all.extend(hypotheses_scores)
    masked_texts_all.extend(masked_texts)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  9.16it/s]


In [114]:
contra_data = dev_data.loc[dev_data['gold_label'] == 'contradiction',].copy()
contra_data['hypothesis_token_pred_binary'] = pred_all
contra_data['hypothesis_token_pred_indexes'] = pred_indexes_all
contra_data['hypothesis_token_pred_scores'] = pred_scores_all

In [115]:
contra_data['hypothesis_word_pred_indexes'] = contra_data.apply(lambda x: get_word_pred_indexes_from_token_pred_indexes(x, 'hypothesis_tok2word', 'hypothesis_token_pred_indexes'), axis=1)
contra_data['hypothesis_word_pred_binary'] = contra_data.apply(lambda x: get_word_pred_from_word_pred_indexes(x, 'hypothesis_words', 'hypothesis_word_pred_indexes'), axis=1)
contra_data['hypothesis_word_pred_scores'] = contra_data.apply(lambda x: get_pred_scores_word(x, 'hypothesis_words', 'hypothesis_token_pred_scores', 'hypothesis_word2tok', method='max'),axis=1)

In [116]:
# ### Calculate Word-level Metrics
contra_data['ap_word']=contra_data.apply(lambda x: apply_ap(x,"hypothesis_word_labels_binary", "hypothesis_word_pred_scores"),axis=1)
contra_data['rr_word']=contra_data.apply(lambda x: get_rr(x,"hypothesis_word_labels_binary", "hypothesis_word_pred_scores"),axis=1)
contra_data['precision_word']=contra_data.apply(lambda x: apply_precision(x,"hypothesis_word_labels_binary", "hypothesis_word_pred_binary"),axis=1)
contra_data['recall_word']=contra_data.apply(lambda x: apply_recall(x,"hypothesis_word_labels_binary", "hypothesis_word_pred_binary"),axis=1)

## Summary metric
mrr = contra_data['rr_word'].mean()
map_score =  contra_data['ap_word'].mean()
precision = contra_data['precision_word'].mean()
recall = contra_data['recall_word'].mean()

In [117]:
# ### Calculate Token-level Metrics
contra_data['ap_tokens']=contra_data.apply(lambda x: apply_ap(x,"hypothesis_token_labels_binary", "hypothesis_token_pred_scores"),axis=1)
contra_data['rr_tokens']=contra_data.apply(lambda x: get_rr(x,"hypothesis_token_labels_binary", "hypothesis_token_pred_scores"),axis=1)
contra_data['precision_tokens']=contra_data.apply(lambda x: apply_precision(x,"hypothesis_token_labels_binary", "hypothesis_token_pred_binary"),axis=1)
contra_data['recall_tokens']=contra_data.apply(lambda x: apply_recall(x,"hypothesis_token_labels_binary", "hypothesis_token_pred_binary"),axis=1)

## Summary metric
mrr_tokens = contra_data['rr_tokens'].mean()
map_score_tokens =  contra_data['ap_tokens'].mean()
precision_tokens = contra_data['precision_tokens'].mean()
recall_tokens = contra_data['recall_tokens'].mean()

In [118]:
print("Metrics evaluated at words level")
print(f"mrr: {mrr:.4f}")
print(f"map: {map_score:.4f}")
print(f"mean precision: {precision:.4f}")
print(f"mean recall: {recall:.4f}")

print("Metrics evaluated at tokens level")
print(f"mrr: {mrr_tokens:.4f}")
print(f"map: {map_score_tokens:.4f}")
print(f"mean precision: {precision_tokens:.4f}")
print(f"mean recall: {recall_tokens:.4f}")

Metrics evaluated at words level
mrr: 0.7806
map: 0.7219
mean precision: 0.4516
mean recall: 0.5346
Metrics evaluated at tokens level
mrr: 0.9140
map: 0.8241
mean precision: 0.6708
mean recall: 0.6124


In [119]:
if criterion is not None:
    contra_data[['pairID', 'hypothesis_word_pred_binary', 'hypothesis_word_pred_scores', 'hypothesis_token_pred_binary', 'hypothesis_token_pred_scores']].to_json(os.path.join(os.path.dirname(model_path), f'EPR_locate_result_{criterion}_{"energy" if use_energy_for_gradient else "proba"}.jsonl'), lines=True, orient='records')
else:
    contra_data[['pairID', 'hypothesis_word_pred_binary', 'hypothesis_word_pred_scores', 'hypothesis_token_pred_binary', 'hypothesis_token_pred_scores']].to_json(os.path.join(os.path.dirname(model_path), f'EPR_locate_result_{"energy" if use_energy_for_gradient else "proba"}.jsonl'), lines=True, orient='records')

In [120]:
metrics_path = os.path.join(os.path.dirname(model_path), 'EPR_metrics.csv')

if not os.path.exists(metrics_path):
    pd.DataFrame({'run_id': [run_id],
                  'criterion': [criterion],
                  'use_energy_for_gradient':[use_energy_for_gradient],
                'classification_accuracy': [acc],
                'mrr_words': [mrr],
                'map_words': [map_score],
                'mean precision_words': [precision],
                'mean recall_words': [recall],
                'mrr_tokens': [mrr_tokens], 
                'map_tokens': [map_score_tokens],
                'mean precision_tokens': [precision_tokens],
                'mean recall_tokens': [recall_tokens]}).to_csv(metrics_path)
else:
    with open(metrics_path, 'a') as f:
        f.write(f"{run_id},{criterion},{use_energy_for_gradient},{acc},{mrr},{map_score},{precision},{recall},{mrr_tokens},{map_score_tokens},{precision_tokens},{recall_tokens}\n")
   

# XY concat model

In [180]:
run_id = 'hayleyson/nli_energynet/v9si3kul'
criterion = 'loss'

In [181]:
## load config
api = wandb.Api()
run = api.run(run_id)
config = run.config
config['device'] = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## load model
model = EncoderModel(config)
model = model.to(config['device'])

if run_id.split('/')[-1] == 'ovp9pnpb': ## exception case handling
    model_path = '/data/hyeryung/loc_edit/' + config['model_path'].split('.pth')[0] + f'_{criterion}.pth'
elif run_id.split('/')[-1] == '8a9jdnfg': ## exception case handling
    model_path = '/data/hyeryung/loc_edit/' + '/'.join(config['model_path'].split('/')[1:]).split('.pth')[0] + f'_{criterion}.pth'
else:
    model_path = config['model_path'].split('.pth')[0] + f'_{criterion}.pth'
print(f"model_path: {model_path}")

try:
    model.load_state_dict(torch.load(model_path,weights_only=True),strict=False)
    config['model_path'] = model_path
except Exception as e:
    model.load_state_dict(torch.load(config['model_path'],weights_only=True),strict=False)
    
## define tokenizer
tokenizer = model.tokenizer

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model_path: /data/hyeryung/loc_edit/models/nli/roberta_large_snli_mnli_anli_train_dev_with_finegrained_finegrained_labels_margin_ranking/1730123591/best_model_loss.pth


## Calculate classification accuracy 

For xy concat model, there are two options.
1. we could find a threshold using dev set and then use that threshold for other datasets.
2. we could calculate energy twice each for the case of label being "consistent" and for "inconsistent" and use a label with lower energy.

I use the second option because it is easier to implement & takes advantage of the training objective we used (to make the energy of correct label lower than the energy of incorrect label.)

In [182]:
## calculate accuracy 
num_correct = 0
labels_binary_all = []
pred_c_binary_all = []
for batch in tqdm.tqdm(all_dataloader):
    premises = [x[0] for x in batch]
    hypotheses = [x[1] for x in batch]
    labels = [x[2] for x in batch]
    labels_binary = torch.IntTensor([1 if lab != 'contradiction' else 0 for lab in labels])
    
    if (config['energynet'].get('input_form', 'x_only') == 'x_only'):  
        sequences = [tokenizer.bos_token + p + tokenizer.sep_token + h + tokenizer.eos_token for p, h in zip(premises, hypotheses)]
        
        tokenized_sequences = tokenizer(sequences, add_special_tokens=False,padding=True, truncation=True, return_tensors='pt')
        tokenized_sequences = tokenized_sequences.to(config['device'])
        logits, hidden_states = model(**tokenized_sequences)
        
    elif (config['energynet'].get('input_form', 'x_only') == 'xy_concat'):
        sequences_cons = [tokenizer.bos_token + p + tokenizer.sep_token + h + tokenizer.sep_token + "consistent" + tokenizer.eos_token for p, h in zip(premises, hypotheses)]
        sequences_incons = [tokenizer.bos_token + p + tokenizer.sep_token + h + tokenizer.sep_token + "inconsistent" + tokenizer.eos_token for p, h in zip(premises, hypotheses)]
        
        tokenized_sequences_cons = tokenizer(sequences_cons, add_special_tokens=False,padding=True, truncation=True, return_tensors='pt')
        tokenized_sequences_cons = tokenized_sequences_cons.to(config['device'])
        logits_cons, hidden_states = model(**tokenized_sequences_cons)
        
        tokenized_sequences_incons = tokenizer(sequences_incons, add_special_tokens=False,padding=True, truncation=True, return_tensors='pt')
        tokenized_sequences_incons = tokenized_sequences_incons.to(config['device'])
        logits_incons, hidden_states = model(**tokenized_sequences_incons)
        
        logits = torch.cat((logits_incons, logits_cons), dim=1)
        # energy is - logits. therefore, logits.argmax(axis=-1) == energies.argmin(axis=-1)
        
    pred_c_binary = logits.argmax(axis=-1).cpu()
    
    num_correct += (labels_binary == pred_c_binary).sum().item()
    pred_c_binary_all.extend(pred_c_binary.tolist())
    labels_binary_all.extend(labels_binary.tolist())
acc = num_correct / len(all_data)
print(acc)


100%|███████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  4.55it/s]

0.94


In [183]:
dev_data['gold_label_binary'] = labels_binary_all
dev_data['pred_binary'] = pred_c_binary_all
if criterion is not None:
    dev_data[['pairID', 'gold_label_binary', 'pred_binary']].to_csv(os.path.join(os.path.dirname(model_path), f'EPR_classification_result_{criterion}.csv'),index=False)
else:
    dev_data[['pairID', 'gold_label_binary', 'pred_binary']].to_csv(os.path.join(os.path.dirname(model_path), 'EPR_classification_result.csv'),index=False)

## Calculate locate performance

In [211]:
# ARGS
use_energy_for_gradient = False
use_consistent_for_label = False
max_num_tokens = 100
input_includes_y = True

In [212]:
locator = LocateMachine(model, model.tokenizer, 'nli')

pred_all = []
pred_indexes_all = []
pred_scores_all = []
masked_texts_all = []
for batch in tqdm.tqdm(contra_dataloader):        
    
    premises = [x[0] for x in batch]
    hypotheses = [x[1] for x in batch]
    if (config['energynet'].get('input_form', 'x_only') == 'x_only'):  
        sequences = [tokenizer.bos_token + p + tokenizer.sep_token + h + tokenizer.eos_token for p, h in zip(premises, hypotheses)]
    elif (config['energynet'].get('input_form', 'x_only') == 'xy_concat'):
        if use_consistent_for_label:
            sequences = [tokenizer.bos_token + p + tokenizer.sep_token + h + tokenizer.sep_token + "consistent" + tokenizer.eos_token for p, h in zip(premises, hypotheses)]
        else:
            sequences = [tokenizer.bos_token + p + tokenizer.sep_token + h + tokenizer.sep_token + "inconsistent" + tokenizer.eos_token for p, h in zip(premises, hypotheses)]
            
    tokenized_sequences = tokenizer(sequences, add_special_tokens=False,padding=True, truncation=True, return_tensors='pt')
    tokenized_sequences = tokenized_sequences.to(config['device'])
    
    masked_texts, scores, label_mask = locator.locate_main(tokenized_sequences, 
                                 'grad_norm', 
                                 max_num_tokens = max_num_tokens, 
                                 unit="word", 
                                 label_id=config['energynet']['energy_col'], 
                                 tokenized_input=True,
                                 return_scores=True,
                                 use_energy=use_energy_for_gradient,
                                 input_includes_y=input_includes_y)
    
    print(label_mask)
    tokenized_result = tokenizer(masked_texts, add_special_tokens=False)
    if (config['energynet'].get('input_form', 'x_only') == 'x_only'): 
        hypotheses_input_ids = [x[x.index(2)+1:-1] for x in tokenized_result['input_ids']]
        hypotheses_scores = [x.tolist()[y.index(2)+1:len(y)-1] for x, y in zip(scores, tokenized_result['input_ids'])]
    elif (config['energynet'].get('input_form', 'x_only') == 'xy_concat'):
        hypotheses_input_ids = [x[x.index(2)+1:x.index(2)+1+x[x.index(2)+1:].index(2)] for x in tokenized_result['input_ids']]
        hypotheses_scores = [x.tolist()[y.index(2)+1:y.index(2)+1+y[y.index(2)+1:].index(2)] for x, y in zip(scores, tokenized_result['input_ids'])]
    
    pred = [[1 if y == tokenizer.mask_token_id else 0 for y in x] for x in hypotheses_input_ids]
    pred_indexes = [np.where(np.array(x) == tokenizer.mask_token_id)[0].tolist() for x in hypotheses_input_ids]
    
    pred_all.extend(pred)
    pred_indexes_all.extend(pred_indexes)
    pred_scores_all.extend(hypotheses_scores)
    masked_texts_all.extend(masked_texts)

  0%|                                                                                                       | 0/1 [00:00<?, ?it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.98it/s]

tensor([[False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True],
        ...,
        [False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True]], device='cuda:0')


In [213]:
contra_data = dev_data.loc[dev_data['gold_label'] == 'contradiction',].copy()
contra_data['hypothesis_token_pred_binary'] = pred_all
contra_data['hypothesis_token_pred_indexes'] = pred_indexes_all
contra_data['hypothesis_token_pred_scores'] = pred_scores_all

In [214]:
contra_data['hypothesis_word_pred_indexes'] = contra_data.apply(lambda x: get_word_pred_indexes_from_token_pred_indexes(x, 'hypothesis_tok2word', 'hypothesis_token_pred_indexes'), axis=1)
contra_data['hypothesis_word_pred_binary'] = contra_data.apply(lambda x: get_word_pred_from_word_pred_indexes(x, 'hypothesis_words', 'hypothesis_word_pred_indexes'), axis=1)
contra_data['hypothesis_word_pred_scores'] = contra_data.apply(lambda x: get_pred_scores_word(x, 'hypothesis_words', 'hypothesis_token_pred_scores', 'hypothesis_word2tok', method='max'),axis=1)

In [215]:
# ### Calculate Word-level Metrics
contra_data['ap_word']=contra_data.apply(lambda x: apply_ap(x,"hypothesis_word_labels_binary", "hypothesis_word_pred_scores"),axis=1)
contra_data['rr_word']=contra_data.apply(lambda x: get_rr(x,"hypothesis_word_labels_binary", "hypothesis_word_pred_scores"),axis=1)
contra_data['precision_word']=contra_data.apply(lambda x: apply_precision(x,"hypothesis_word_labels_binary", "hypothesis_word_pred_binary"),axis=1)
contra_data['recall_word']=contra_data.apply(lambda x: apply_recall(x,"hypothesis_word_labels_binary", "hypothesis_word_pred_binary"),axis=1)

## Summary metric
mrr = contra_data['rr_word'].mean()
map_score =  contra_data['ap_word'].mean()
precision = contra_data['precision_word'].mean()
recall = contra_data['recall_word'].mean()

In [216]:
# ### Calculate Token-level Metrics
contra_data['ap_tokens']=contra_data.apply(lambda x: apply_ap(x,"hypothesis_token_labels_binary", "hypothesis_token_pred_scores"),axis=1)
contra_data['rr_tokens']=contra_data.apply(lambda x: get_rr(x,"hypothesis_token_labels_binary", "hypothesis_token_pred_scores"),axis=1)
contra_data['precision_tokens']=contra_data.apply(lambda x: apply_precision(x,"hypothesis_token_labels_binary", "hypothesis_token_pred_binary"),axis=1)
contra_data['recall_tokens']=contra_data.apply(lambda x: apply_recall(x,"hypothesis_token_labels_binary", "hypothesis_token_pred_binary"),axis=1)

## Summary metric
mrr_tokens = contra_data['rr_tokens'].mean()
map_score_tokens =  contra_data['ap_tokens'].mean()
precision_tokens = contra_data['precision_tokens'].mean()
recall_tokens = contra_data['recall_tokens'].mean()

In [217]:
print("Metrics evaluated at words level")
print(f"mrr: {mrr:.4f}")
print(f"map: {map_score:.4f}")
print(f"mean precision: {precision:.4f}")
print(f"mean recall: {recall:.4f}")

print("Metrics evaluated at tokens level")
print(f"mrr: {mrr_tokens:.4f}")
print(f"map: {map_score_tokens:.4f}")
print(f"mean precision: {precision_tokens:.4f}")
print(f"mean recall: {recall_tokens:.4f}")

Metrics evaluated at words level
mrr: 0.7417
map: 0.6845
mean precision: 0.4818
mean recall: 0.5540
Metrics evaluated at tokens level
mrr: 0.8737
map: 0.7747
mean precision: 0.7016
mean recall: 0.6199


In [218]:
if criterion is not None:
    contra_data[['pairID', 'hypothesis_word_pred_binary', 'hypothesis_word_pred_scores', 'hypothesis_token_pred_binary', 'hypothesis_token_pred_scores']].to_json(os.path.join(os.path.dirname(model_path), f'EPR_locate_result_{criterion}_{"energy" if use_energy_for_gradient else "proba"}_{"cons_as_label" if use_consistent_for_label else "incons_as_label"}.jsonl'), lines=True, orient='records')
else:
    contra_data[['pairID', 'hypothesis_word_pred_binary', 'hypothesis_word_pred_scores', 'hypothesis_token_pred_binary', 'hypothesis_token_pred_scores']].to_json(os.path.join(os.path.dirname(model_path), f'EPR_locate_result_{"energy" if use_energy_for_gradient else "proba"}_{"cons_as_label" if use_consistent_for_label else "incons_as_label"}.jsonl'), lines=True, orient='records')

In [219]:
metrics_path = os.path.join(os.path.dirname(model_path), 'EPR_metrics.csv')

if not os.path.exists(metrics_path):
    pd.DataFrame({'run_id': [run_id],
                  'criterion': [criterion],
                  'use_energy_for_gradient':[use_energy_for_gradient],
                  'use_consistent_for_label':[use_consistent_for_label],
                'classification_accuracy': [acc],
                'mrr_words': [mrr],
                'map_words': [map_score],
                'mean precision_words': [precision],
                'mean recall_words': [recall],
                'mrr_tokens': [mrr_tokens], 
                'map_tokens': [map_score_tokens],
                'mean precision_tokens': [precision_tokens],
                'mean recall_tokens': [recall_tokens]}).to_csv(metrics_path,index=False)
else:
    with open(metrics_path, 'a') as f:
        f.write(f"{run_id},{criterion},{use_energy_for_gradient},{use_consistent_for_label},{acc},{mrr},{map_score},{precision},{recall},{mrr_tokens},{map_score_tokens},{precision_tokens},{recall_tokens}\n")
   